In [89]:
## Random Forest Implementation
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

In [90]:
df = pd.read_csv('Travel.csv')
df.head()

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,3.0,Deluxe,3.0,Single,1.0,1,2,1,0.0,Manager,20993.0
1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,3,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,4.0,Basic,3.0,Single,7.0,1,3,0,0.0,Executive,17090.0
3,200003,0,33.0,Company Invited,1,9.0,Salaried,Female,2,3.0,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,200004,0,NaN,Self Enquiry,1,8.0,Small Business,Male,2,3.0,Basic,4.0,Divorced,1.0,0,5,1,0.0,Executive,18468.0


In [91]:
df.isnull().sum()

CustomerID                    0
ProdTaken                     0
Age                         226
TypeofContact                25
CityTier                      0
DurationOfPitch             251
Occupation                    0
Gender                        0
NumberOfPersonVisiting        0
NumberOfFollowups            45
ProductPitched                0
PreferredPropertyStar        26
MaritalStatus                 0
NumberOfTrips               140
Passport                      0
PitchSatisfactionScore        0
OwnCar                        0
NumberOfChildrenVisiting     66
Designation                   0
MonthlyIncome               233
dtype: int64

In [92]:
df['Gender'].value_counts()

Gender
Male       2916
Female     1817
Fe Male     155
Name: count, dtype: int64

In [93]:
df['Gender'] = df['Gender'].replace('Fe Male', 'Female')

In [94]:
df['Gender'].value_counts()

Gender
Male      2916
Female    1972
Name: count, dtype: int64

In [95]:
df['MaritalStatus'].value_counts()

MaritalStatus
Married      2340
Divorced      950
Single        916
Unmarried     682
Name: count, dtype: int64

In [96]:
df['TypeofContact'].value_counts()

TypeofContact
Self Enquiry       3444
Company Invited    1419
Name: count, dtype: int64

In [97]:
df['MaritalStatus'] = df['MaritalStatus'].replace('Single', 'Unmarried')

In [98]:
df['MaritalStatus'].value_counts()

MaritalStatus
Married      2340
Unmarried    1598
Divorced      950
Name: count, dtype: int64

In [99]:
df.head()

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,3.0,Deluxe,3.0,Unmarried,1.0,1,2,1,0.0,Manager,20993.0
1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,3,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,4.0,Basic,3.0,Unmarried,7.0,1,3,0,0.0,Executive,17090.0
3,200003,0,33.0,Company Invited,1,9.0,Salaried,Female,2,3.0,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,200004,0,NaN,Self Enquiry,1,8.0,Small Business,Male,2,3.0,Basic,4.0,Divorced,1.0,0,5,1,0.0,Executive,18468.0


In [100]:
## check missing values
features_with_na = [features for features in df.columns if df[features].isnull().sum() >= 1]
for feature in features_with_na:
    print(feature, np.round(df[feature].isnull().mean()*100,3), 'missing value')

Age 4.624 missing value
TypeofContact 0.511 missing value
DurationOfPitch 5.135 missing value
NumberOfFollowups 0.921 missing value
PreferredPropertyStar 0.532 missing value
NumberOfTrips 2.864 missing value
NumberOfChildrenVisiting 1.35 missing value
MonthlyIncome 4.767 missing value


In [101]:
## statistics for numerical column
df[features_with_na].select_dtypes(exclude='object').describe()

,Age,DurationOfPitch,NumberOfFollowups,PreferredPropertyStar,NumberOfTrips,NumberOfChildrenVisiting,MonthlyIncome
count,4662.000000,4637.000000,4843.000000,4862.000000,4748.000000,4822.000000,4655.000000
mean,37.622265,15.490835,3.708445,3.581037,3.236521,1.187267,23619.853491
std,9.316387,8.519643,1.002509,0.798009,1.849019,0.857861,5380.698361
min,18.000000,5.000000,1.000000,3.000000,1.000000,0.000000,1000.000000
25%,31.000000,9.000000,3.000000,3.000000,2.000000,1.000000,20346.000000
50%,36.000000,13.000000,4.000000,3.000000,3.000000,1.000000,22347.000000
75%,44.000000,20.000000,4.000000,4.000000,4.000000,2.000000,25571.000000
max,61.000000,127.000000,6.000000,5.000000,22.000000,3.000000,98678.000000


## Imputing Null values
1. Impute Median value for Age column
2. Impute Mode for Type of Contract
3. Impute Median for Duration of Pitch
4. Impute Mode for NumberofFollowup as it is Discrete feature
5. Impute Mode for PreferredPropertyStar
6. Impute Median for NumberofTrips
7. Impute Mode for NumberOfChildrenVisiting
8. Impute Median for MonthlyIncome

In [102]:
df['Age'].fillna(df['Age'].mean(), inplace=True)
df.TypeofContact.fillna(df.TypeofContact.mode()[0], inplace=True)
df.DurationOfPitch.fillna(df.DurationOfPitch.median(), inplace=True) ##dot notation work only when column name has no space

#NumberOfFollowups
df.NumberOfFollowups.fillna(df.NumberOfFollowups.mode()[0], inplace=True)

#PreferredPropertyStar
df.PreferredPropertyStar.fillna(df.PreferredPropertyStar.mode()[0], inplace=True)

#NumberOfTrips
df.NumberOfTrips.fillna(df.NumberOfTrips.median(), inplace=True)

#NumberOfChildrenVisiting
df.NumberOfChildrenVisiting.fillna(df.NumberOfChildrenVisiting.mode()[0], inplace=True)

#MonthlyIncome
df.MonthlyIncome.fillna(df.MonthlyIncome.median(), inplace=True)

In [103]:
df.isnull().sum()

CustomerID                  0
ProdTaken                   0
Age                         0
TypeofContact               0
CityTier                    0
DurationOfPitch             0
Occupation                  0
Gender                      0
NumberOfPersonVisiting      0
NumberOfFollowups           0
ProductPitched              0
PreferredPropertyStar       0
MaritalStatus               0
NumberOfTrips               0
Passport                    0
PitchSatisfactionScore      0
OwnCar                      0
NumberOfChildrenVisiting    0
Designation                 0
MonthlyIncome               0
dtype: int64

In [104]:
df.drop('CustomerID', inplace=True, axis=1)

In [105]:
## Feature Selection
df.head()

,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,1,41.000000,Self Enquiry,3,6.0,Salaried,Female,3,3.0,Deluxe,3.0,Unmarried,1.0,1,2,1,0.0,Manager,20993.0
1,0,49.000000,Company Invited,1,14.0,Salaried,Male,3,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,1,37.000000,Self Enquiry,1,8.0,Free Lancer,Male,3,4.0,Basic,3.0,Unmarried,7.0,1,3,0,0.0,Executive,17090.0
3,0,33.000000,Company Invited,1,9.0,Salaried,Female,2,3.0,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,0,37.622265,Self Enquiry,1,8.0,Small Business,Male,2,3.0,Basic,4.0,Divorced,1.0,0,5,1,0.0,Executive,18468.0


In [106]:
df['TotalVisiting'] = df['NumberOfChildrenVisiting'] + df['NumberOfPersonVisiting']
df.drop(columns=['NumberOfChildrenVisiting','NumberOfPersonVisiting'], inplace=True, axis=1)

In [107]:
df.head()

,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,Designation,MonthlyIncome,TotalVisiting
0,1,41.000000,Self Enquiry,3,6.0,Salaried,Female,3.0,Deluxe,3.0,Unmarried,1.0,1,2,1,Manager,20993.0,3.0
1,0,49.000000,Company Invited,1,14.0,Salaried,Male,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,Manager,20130.0,5.0
2,1,37.000000,Self Enquiry,1,8.0,Free Lancer,Male,4.0,Basic,3.0,Unmarried,7.0,1,3,0,Executive,17090.0,3.0
3,0,33.000000,Company Invited,1,9.0,Salaried,Female,3.0,Basic,3.0,Divorced,2.0,1,5,1,Executive,17909.0,3.0
4,0,37.622265,Self Enquiry,1,8.0,Small Business,Male,3.0,Basic,4.0,Divorced,1.0,0,5,1,Executive,18468.0,2.0


In [108]:
num_feat = [feature for feature in df.columns if df[feature].dtype != 'O']
cat_feat = [feature for feature in df.columns if df[feature].dtype == 'O']  ## dot notation when accesing column and [] for iterating column
dis_feat = [feature for feature in num_feat if len(df[feature].unique()) <= 25]
cont_feat = [feature for feature in num_feat if feature not in dis_feat]

print('No. of feature in', len(num_feat))
print('No. of feature in', len(cat_feat))
print('No. of feature in', len(dis_feat))
print('No. of feature in', len(cont_feat))


No. of feature in 12
No. of feature in 6
No. of feature in 9
No. of feature in 3


In [109]:
df.head()

,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,Designation,MonthlyIncome,TotalVisiting
0,1,41.000000,Self Enquiry,3,6.0,Salaried,Female,3.0,Deluxe,3.0,Unmarried,1.0,1,2,1,Manager,20993.0,3.0
1,0,49.000000,Company Invited,1,14.0,Salaried,Male,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,Manager,20130.0,5.0
2,1,37.000000,Self Enquiry,1,8.0,Free Lancer,Male,4.0,Basic,3.0,Unmarried,7.0,1,3,0,Executive,17090.0,3.0
3,0,33.000000,Company Invited,1,9.0,Salaried,Female,3.0,Basic,3.0,Divorced,2.0,1,5,1,Executive,17909.0,3.0
4,0,37.622265,Self Enquiry,1,8.0,Small Business,Male,3.0,Basic,4.0,Divorced,1.0,0,5,1,Executive,18468.0,2.0


In [110]:
## independent and dependnt feature
X = df.drop(['ProdTaken'],axis=1 )
y = df['ProdTaken']

In [111]:
X.shape

(4888, 17)

In [112]:
## Model Trainng
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.33,random_state=42)

In [113]:
X_train.shape

(3274, 17)

In [114]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3274 entries, 2962 to 860
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Age                     3274 non-null   float64
 1   TypeofContact           3274 non-null   object 
 2   CityTier                3274 non-null   int64  
 3   DurationOfPitch         3274 non-null   float64
 4   Occupation              3274 non-null   object 
 5   Gender                  3274 non-null   object 
 6   NumberOfFollowups       3274 non-null   float64
 7   ProductPitched          3274 non-null   object 
 8   PreferredPropertyStar   3274 non-null   float64
 9   MaritalStatus           3274 non-null   object 
 10  NumberOfTrips           3274 non-null   float64
 11  Passport                3274 non-null   int64  
 12  PitchSatisfactionScore  3274 non-null   int64  
 13  OwnCar                  3274 non-null   int64  
 14  Designation             3274 non-null   obj

In [115]:
## create column transformer wit 3 one hot encoder

cat_feat = X.select_dtypes(include='object').columns
num_feat = X.select_dtypes(exclude='object').columns

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

num_transformer = StandardScaler()
oh_transformer = OneHotEncoder(drop='first')

preprocessor = ColumnTransformer(
    [
        ('OneHotEncoder', oh_transformer, cat_feat),
        ('StandardScaler', num_transformer, num_feat)
    ]
)


In [116]:
preprocessor

,transformers,"[('OneHotEncoder', ...), ('StandardScaler', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,'first'
,sparse_output,True


In [117]:
X_train = preprocessor.fit_transform(X_train)

In [118]:
X_test = preprocessor.transform(X_test)

In [119]:
## Random forest Classifier technique
pd.DataFrame(X_train)

,0,1,2,3,4,5,6,7,8,9,...,16,17,18,19,20,21,22,23,24,25
0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,...,1.483420,-0.161387,1.280351,-0.725158,-0.674414,-0.637936,-0.776562,0.783328,-0.160488,0.635436
1,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.708239,-0.527642,-0.721975,0.527040,-0.674414,1.567556,-0.047645,-1.276604,-1.139309,-0.780778
2,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,...,-0.708239,0.815290,0.279188,1.779238,-0.130564,-0.637936,0.681272,0.783328,2.269527,2.051650
3,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,...,-0.708239,2.036137,0.279188,-0.725158,-0.130564,1.567556,-0.047645,0.783328,1.030274,0.635436
4,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.708239,-1.015980,1.280351,0.527040,2.588687,-0.637936,-0.047645,0.783328,-0.478024,0.635436
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3269,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,-0.708239,-0.649726,1.280351,-0.725158,-0.674414,-0.637936,-1.505478,0.783328,-0.531758,0.635436
3270,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,1.483420,-0.893896,-0.721975,1.779238,-1.218264,-0.637936,1.410189,0.783328,1.503770,-0.072671
3271,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.483420,1.547798,0.279188,-0.725158,2.044837,-0.637936,-0.776562,0.783328,-0.358012,0.635436
3272,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,...,1.483420,1.791968,1.280351,-0.725158,-0.130564,-0.637936,-1.505478,0.783328,-0.251854,0.635436


In [120]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score,classification_report, roc_curve, roc_auc_score, \
                precision_score, f1_score, recall_score

In [121]:
models = {
    'Logistic Regression' : LogisticRegression(),
    'Decision Tree' : DecisionTreeClassifier(),
    'Gradient Boost' : GradientBoostingClassifier(),
    'Random Forest' : RandomForestClassifier()
}

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train,y_train)

    ## make prediction
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    ## Model Test Performance
    model_train_accuracy = accuracy_score(y_test, y_test_pred)
    model_train_f1 = f1_score(y_test,y_test_pred,average='weighted')
    model_train_precision = precision_score(y_test, y_test_pred)
    model_train_recall = recall_score(y_test, y_test_pred)
    model_train_rocauc_score = roc_auc_score(y_test, y_test_pred)

    
    # Test set performance
    model_test_accuracy = accuracy_score(y_test, y_test_pred) # Calculate Accuracy
    model_test_f1 = f1_score(y_test, y_test_pred, average='weighted') # Calculate F1-score
    model_test_precision = precision_score(y_test, y_test_pred) # Calculate Precision
    model_test_recall = recall_score(y_test, y_test_pred) # Calculate Recall
    model_test_rocauc_score = roc_auc_score(y_test, y_test_pred) #Calculate Roc

    print(list(models.keys())[i])

    print('Train Test Accuracy')
    print('- Accuracy {:.4f}'.format(model_train_accuracy))
    print('- F1 score {:.4f}'.format(model_train_f1))
    print('- Precission {:.4f}'.format(model_train_precision))
    print('- recall {:.4f}'.format(model_train_recall))
    print('- roc auc score {:.4f}'.format(model_train_rocauc_score))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print('- Accuracy: {:.4f}'.format(model_test_accuracy))
    print('- F1 score: {:.4f}'.format(model_test_f1))
    print('- Precision: {:.4f}'.format(model_test_precision))
    print('- Recall: {:.4f}'.format(model_test_recall))
    print('- Roc Auc Score: {:.4f}'.format(model_test_rocauc_score))

    print('--'*35)
    print('/')

Logistic Regression
Train Test Accuracy
- Accuracy 0.8439
- F1 score 0.8208
- Precission 0.6454
- recall 0.3106
- roc auc score 0.6364
----------------------------------
Model performance for Test set
- Accuracy: 0.8439
- F1 score: 0.8208
- Precision: 0.6454
- Recall: 0.3106
- Roc Auc Score: 0.6364
----------------------------------------------------------------------
/
Decision Tree
Train Test Accuracy
- Accuracy 0.8916
- F1 score 0.8933
- Precission 0.6855
- recall 0.7440
- roc auc score 0.8342
----------------------------------
Model performance for Test set
- Accuracy: 0.8916
- F1 score: 0.8933
- Precision: 0.6855
- Recall: 0.7440
- Roc Auc Score: 0.8342
----------------------------------------------------------------------
/
Gradient Boost
Train Test Accuracy
- Accuracy 0.8680
- F1 score 0.8505
- Precission 0.7632
- recall 0.3959
- roc auc score 0.6843
----------------------------------
Model performance for Test set
- Accuracy: 0.8680
- F1 score: 0.8505
- Precision: 0.7632
- Reca

In [122]:
## hyperparameter tuning
params = {
    'n_estimators' : [100, 300, 500, 1000],
    'max_depth': [5, 8, 15, None, 10],
    "max_features": [5, 7, "auto", 8],
    "min_samples_split": [2, 8, 15, 20],
}

In [123]:
randomcv_models = [
    ('RF', RandomForestClassifier(),params)
]

In [124]:
from sklearn.model_selection import RandomizedSearchCV
model_param = {}

for name,models,params in randomcv_models:
    random = RandomizedSearchCV(
                                estimator=models,
                                   param_distributions=params,
                                   n_iter=100,
                                   cv=3,
                                   verbose=2,
                                   n_jobs=-1)
    random.fit(X_train, y_train)
    model_param[name] = random.best_params_

for model_name in model_param:
    print(f"---------------- Best Params for {model_name} -------------------")
    print(model_param[model_name])



Fitting 3 folds for each of 100 candidates, totalling 300 fits
---------------- Best Params for RF -------------------
{'n_estimators': 500, 'min_samples_split': 2, 'max_features': 8, 'max_depth': 15}


In [125]:
models = {
    'Random Forest' : RandomForestClassifier(n_estimators = 300, min_samples_split = 2, max_features = 8, max_depth = 15)
}

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train,y_train)

    ## make prediction
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    ## Model Test Performance
    model_train_accuracy = accuracy_score(y_test, y_test_pred)
    model_train_f1 = f1_score(y_test,y_test_pred,average='weighted')
    model_train_precision = precision_score(y_test, y_test_pred)
    model_train_recall = recall_score(y_test, y_test_pred)
    model_train_rocauc_score = roc_auc_score(y_test, y_test_pred)

    
    # Test set performance
    model_test_accuracy = accuracy_score(y_test, y_test_pred) # Calculate Accuracy
    model_test_f1 = f1_score(y_test, y_test_pred, average='weighted') # Calculate F1-score
    model_test_precision = precision_score(y_test, y_test_pred) # Calculate Precision
    model_test_recall = recall_score(y_test, y_test_pred) # Calculate Recall
    model_test_rocauc_score = roc_auc_score(y_test, y_test_pred) #Calculate Roc

    print(list(models.keys())[i])

    print('Train Test Accuracy')
    print('- Accuracy {:.4f}'.format(model_train_accuracy))
    print('- F1 score {:.4f}'.format(model_train_f1))
    print('- Precission {:.4f}'.format(model_train_precision))
    print('- recall {:.4f}'.format(model_train_recall))
    print('- roc auc score {:.4f}'.format(model_train_rocauc_score))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print('- Accuracy: {:.4f}'.format(model_test_accuracy))
    print('- F1 score: {:.4f}'.format(model_test_f1))
    print('- Precision: {:.4f}'.format(model_test_precision))
    print('- Recall: {:.4f}'.format(model_test_recall))
    print('- Roc Auc Score: {:.4f}'.format(model_test_rocauc_score))

    print('--'*35)
    print('/')

Random Forest
Train Test Accuracy
- Accuracy 0.9232
- F1 score 0.9179
- Precission 0.9043
- recall 0.6451
- roc auc score 0.8150
----------------------------------
Model performance for Test set
- Accuracy: 0.9232
- F1 score: 0.9179
- Precision: 0.9043
- Recall: 0.6451
- Roc Auc Score: 0.8150
----------------------------------------------------------------------
/
